# Extract DINOv3 embeddings

Letterbox-resizes house images (preserving aspect ratio, padding to a square) and extracts frozen DINOv3 embeddings — CLS token concatenated with a **masked** mean-pool over patch tokens, so padding patches don't dilute the pooled feature.

**Expects:**
- `dataset.py` in the same directory (defines `HouseDataset`)
- `house_dataset/train/<id>.jpg`
- `house_dataset/train.csv`
- `dinov3/` — a local clone of https://github.com/facebookresearch/dinov3 (or use `source="github"` in the model-loading cell)
- `dinov3_ViT_weights/<checkpoint>.pth` — downloaded checkpoint


In [1]:
import os
import numpy as np
import torch
from torchvision.transforms import v2
import torchvision.transforms.functional as F
from tqdm import tqdm

from dataset import HouseDataset

## Config

In [2]:
REPO_DIR = "dinov3"
MODEL_NAME = "dinov3_vits16plus"
weights_directory = "dinov3_ViT_weights"
weights_filename = "dinov3_vits16plus_pretrain_lvd1689m-4057cbaa"
WEIGHTS = os.path.join(weights_directory, weights_filename + ".pth")

IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

## Letterbox transform with patch mask

Resizes so the long side hits `IMG_SIZE` (no stretching), pads the short side with neutral gray to fill the square, and returns a per-patch validity mask (`True` = real content, `False` = padding) at DINOv3's 16×16 patch-grid resolution — used later to mean-pool only over patches that actually contain house pixels.

In [3]:
class LetterboxResize(torch.nn.Module):
    def __init__(self, size=224, patch_size=16, fill=114):
        super().__init__()
        self.size = size
        self.patch_size = patch_size
        self.fill = fill  # 114 = neutral gray, same convention as YOLO's letterbox

    def forward(self, img):
        _, h, w = img.shape
        scale = self.size / max(h, w)
        new_h, new_w = round(h * scale), round(w * scale)
        img = F.resize(img, [new_h, new_w], interpolation=F.InterpolationMode.BICUBIC)

        pad_h, pad_w = self.size - new_h, self.size - new_w
        top, bottom = pad_h // 2, pad_h - pad_h // 2
        left, right = pad_w // 2, pad_w - pad_w // 2
        img = F.pad(img, [left, top, right, bottom], fill=self.fill)

        # patch-grid mask: mark a patch valid if it overlaps the real (non-padded) region at all
        grid = self.size // self.patch_size
        mask = torch.zeros(grid, grid, dtype=torch.bool)
        r0, r1 = top // self.patch_size, -(-(top + new_h) // self.patch_size)   # ceil div for end
        c0, c1 = left // self.patch_size, -(-(left + new_w) // self.patch_size)
        mask[r0:r1, c0:c1] = True
        return img, mask.flatten()   # row-major, matches ViT patch token ordering


class MaskedLetterboxTransform(torch.nn.Module):
    """LetterboxResize followed by ToDtype/Normalize on the image only — the mask passes
    through unchanged, since v2.Compose would otherwise try to feed the (image, mask) tuple
    into the next transform as if it were a plain image tensor."""
    def __init__(self, size, mean, std):
        super().__init__()
        self.letterbox = LetterboxResize(size)
        self.normalize = v2.Compose([
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])

    def forward(self, img):
        img, mask = self.letterbox(img)
        img = self.normalize(img)
        return img, mask

## Device

In [4]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("using device:", device)

using device: mps


## Dataset + dataloader

In [5]:
transform = MaskedLetterboxTransform(IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD)

training_dataset = HouseDataset("house_dataset/train", "house_dataset/train.csv")
training_dataset.set_transform(transform)
loader = training_dataset.get_dataloader(batch_size=16, shuffle=False)

## Load frozen DINOv3 model

In [6]:
MODEL_NAME = "dinov3_vith16plus"
weights_directory = "dinov3_ViT_weights"
weights_filename = "dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth"
WEIGHTS = os.path.join(weights_directory, weights_filename)

model = torch.hub.load(
    "facebookresearch/dinov3", MODEL_NAME, source="github", 
    weights=WEIGHTS
)
model = model.to(device).eval()

Using cache found in /Users/act133/.cache/torch/hub/facebookresearch_dinov3_main


## Extract embeddings\n\nCLS token concatenated with a mask-weighted mean of patch tokens (padding patches excluded).

In [7]:
all_embeddings, all_prices, all_filenames = [], [], []

with torch.no_grad():
    for imgs, masks, prices, fnames in tqdm(loader):
        imgs = imgs.to(device)
        masks = masks.to(device)                               # (B, num_patches), bool

        features = model.forward_features(imgs)
        cls = features["x_norm_clstoken"]                       # (B, dim)
        patch_tokens = features["x_norm_patchtokens"]           # (B, num_patches, dim)

        # mean-pool over real-content patches only, excluding padding
        mask_f = masks.unsqueeze(-1).float()                    # (B, num_patches, 1)
        patch_mean = (patch_tokens * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)

        embedding = torch.cat([cls, patch_mean], dim=1)         # (B, 2*dim)

        all_embeddings.append(embedding.cpu().numpy())
        all_prices.append(prices.numpy())
        all_filenames.extend(fnames)

100%|████████████████████████████████████████████████████████████████████████████████| 500/500 [16:40<00:00,  2.00s/it]


## Save

In [8]:
X = np.concatenate(all_embeddings)
y = np.concatenate(all_prices)

np.save("train_embeddings.npy", X)
np.save("train_prices.npy", y)
print("saved embeddings:", X.shape, "prices:", y.shape)

saved embeddings: (8000, 2560) prices: (8000,)
